# HighPitchCT — Stage 1 على Kaggle (`v0.4.0`): تدقيق الهندسة الحلزونية ثلاثية الأبعاد، البؤرة المتحركة `FFS`، البراهين الفيزيائية للتوجيه، والفانتوم التحليلي

1. فعّل **Internet = On** من إعدادات الدفتر.
2. أضف سرًا في **Add-ons → Secrets** باسم `GITHUB_TOKEN`.
3. يكفي `Accelerator = None` في `Stage 1` (أو `GPU T4 x2` إن رغبت؛ القراءة والبراهين الهندسية سريعة جدًا على المعالج وتستغرق ثواني معدودة بذاكرة أقل من `100 MiB`).
4. إذا كانت بيانات `C012` موجودة مسبقًا في `/kaggle/working/highpitchct-data` فلن يُعاد تنزيلها، وإلا سينزّلها تلقائيًا ضمن سقف `<= 18.0 GiB` ثم يشغّل تدقيق `Stage 1` الكامل.


In [ ]:
import os
from pathlib import Path
from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret('GITHUB_TOKEN')
repo = Path('/kaggle/working/HighPitchCT-Pipeline')
if not repo.exists():
    rc = os.system(f'git clone https://{token}@github.com/BrahimSoufghalem/HighPitchCT-Pipeline.git {repo}')
    assert rc == 0, 'فشل استنساخ المستودع؛ تحقق من Internet ومن GITHUB_TOKEN'
else:
    rc = os.system(f'git -C {repo} pull --ff-only')
    assert rc == 0, 'فشل تحديث المستودع عبر git pull'

%cd /kaggle/working/HighPitchCT-Pipeline
!python -m pip install -q -e .
!python -m unittest discover -s tests -t . -v


## 1. التأكد من جاهزية بيانات `C012` وتشغيل التدقيق الفيزيائي الكامل `hpct stage1-audit`


In [ ]:
!hpct download --series-file configs/pilot_c012.csv --workers 4 --retries 5
!hpct stage1-audit --case-id C012 --run-phantom-check --workers 8


## 2. عرض جدول البراهين الفيزيائية الثلاثة على بيانات `C012` الحقيقية والفانتوم التحليلي


In [ ]:
import json
import pandas as pd

report_path = '/kaggle/working/highpitchct-data/manifests/stage1_audit_report.json'
with open(report_path, encoding='utf-8') as f:
    rep = json.load(f)

c012 = rep['audited_series'][0]
ov = c012['orientation_verification']
conj = ov['helical_conjugate_ray_proof']
ft = ov['full_turn_axial_overlap_proof']

summary_df = pd.DataFrame([
    {
        'الاختبار الفيزيائي': '1) نعومة التدرج المحوري (Fortran vs C)',
        'الفرضية الصحيحة': ov['layout_proof']['mean_axial_diff_fortran'],
        'الفرضية الخاطئة': ov['layout_proof']['mean_axial_diff_c_order'],
        'النجاح': ov['layout_proof']['pass'],
    },
    {
        'الاختبار الفيزيائي': '2أ) الأشعة المترافقة الحلزونية: اتجاه الصفوف (row 0 = +z vs -z)',
        'الفرضية الصحيحة': conj.get('mae_true_geometry'),
        'الفرضية الخاطئة': conj.get('mae_flipped_axial_rows'),
        'النجاح': conj.get('axial_row_sign_pass'),
    },
    {
        'الاختبار الفيزيائي': '2ب) الأشعة المترافقة الحلزونية: اتجاه الأعمدة (+e_u vs -e_u)',
        'الفرضية الصحيحة': conj.get('mae_true_geometry'),
        'الفرضية الخاطئة': conj.get('mae_flipped_detector_cols'),
        'النجاح': conj.get('transverse_col_and_c0_pass'),
    },
    {
        'الاختبار الفيزيائي': '2ج) الأشعة المترافقة الحلزونية: طرح 1.0 من مركز الكاشف (0-based vs 1-based)',
        'الفرضية الصحيحة': conj.get('mae_true_geometry'),
        'الفرضية الخاطئة': conj.get('mae_unconverted_1based_c0'),
        'النجاح': conj.get('transverse_col_and_c0_pass'),
    },
    {
        'الاختبار الفيزيائي': '3) تداخل الدورة الكاملة 2π عند مركز الدوران (p < 0.96)',
        'الفرضية الصحيحة': ft.get('mae_true_row0_plus_z'),
        'الفرضية الخاطئة': ft.get('mae_flipped_row0_minus_z'),
        'النجاح': ft.get('pass'),
    },
])
display(summary_df)


## 3. معاينة بصرية لإسقاطات `C012` الحقيقية والفانتوم التحليلي ثلاثي الأبعاد للصدر


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
from highpitchct.ctpd_reader import read_single_projection, scan_projection_series
from highpitchct.phantom import build_thorax_phantom, voxelize_phantom

geom, files = scan_projection_series(c012['series_dir'], workers=8)
mid_view = read_single_projection(files[len(files) // 2])

phantom = build_thorax_phantom()
vox = voxelize_phantom(phantom, shape_zyx=(96, 192, 192), spacing_zyx_mm=(2.5, 1.8, 1.8))
hu = vox['hu_volume']

fig, axes = plt.subplots(1, 3, figsize=(16, 4.5))
im0 = axes[0].imshow(mid_view, aspect='auto', cmap='bone')
axes[0].set_title(f'C012 Mid Projection {mid_view.shape} [row, col]')
axes[0].set_xlabel('Detector Column (0..735)')
axes[0].set_ylabel('Detector Row (0=+z .. 63=-z)')
plt.colorbar(im0, ax=axes[0], fraction=0.046)

im1 = axes[1].imshow(hu[hu.shape[0] // 2], cmap='gray', vmin=-1000, vmax=400, origin='lower')
axes[1].set_title('3D Thorax Phantom — Axial Mid Slice [HU]')
plt.colorbar(im1, ax=axes[1], fraction=0.046)

im2 = axes[2].imshow(hu[:, hu.shape[1] // 2, :], cmap='gray', vmin=-1000, vmax=400, origin='lower', aspect='auto')
axes[2].set_title('3D Thorax Phantom — Coronal Slice [HU]')
plt.colorbar(im2, ax=axes[2], fraction=0.046)
plt.tight_layout()
plt.show()
